In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

from src.allocator import replace_linear_with_lplora, LpLoRAAllocator

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Connected to: {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'}")

# Global dictionary to store all logs across different models and tasks
ALL_EXPERIMENT_LOGS = {}

In [ ]:
TASK_CONFIG = {
    "cola": {"keys": ("sentence", None), "metric": "matthews_correlation", "num_labels": 2, "is_reg": False},
    "mrpc": {"keys": ("sentence1", "sentence2"), "metric": "accuracy", "num_labels": 2, "is_reg": False},
    "rte":  {"keys": ("sentence1", "sentence2"), "metric": "accuracy", "num_labels": 2, "is_reg": False},
    "stsb": {"keys": ("sentence1", "sentence2"), "metric": "pearson", "num_labels": 1, "is_reg": True},
    "sst2": {"keys": ("sentence", None), "metric": "accuracy", "num_labels": 2, "is_reg": False},
}

def run_lplora_experiment(
    model_name: str = "microsoft/deberta-v3-base",
    task: str = "mrpc",
    init_r: int = 16,
    target_ratio: float = 0.5,
    p: float = 0.5,
    M: float = 10.0,
    lr: float = 8e-4,
    batch_size: int = 32,
    epochs: int = 10,
    max_len: int = 256,
    seed: int = 42,
):
    run_id = f"{model_name.split('/')[-1]}_{task}_r{init_r}_ratio{target_ratio}_p{p}"
    print(f"\n{'='*75}\nStarting Run: {run_id}\n{'='*75}")

    torch.manual_seed(seed)
    np.random.seed(seed)

    cfg = TASK_CONFIG[task]
    raw_datasets = load_dataset("glue", task)
    metric = evaluate.load("glue", task)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    s1_key, s2_key = cfg["keys"]
    def preprocess_fn(examples):
        texts = (examples[s1_key],) if s2_key is None else (examples[s1_key], examples[s2_key])
        res = tokenizer(*texts, truncation=True, max_length=max_len)
        res["labels"] = examples["label"]
        return res

    tokenized = raw_datasets.map(
        preprocess_fn, batched=True, remove_columns=raw_datasets["train"].column_names
    )
    collator = DataCollatorWithPadding(tokenizer)
    train_loader = DataLoader(tokenized["train"], batch_size=batch_size, shuffle=True, collate_fn=collator)
    val_loader = DataLoader(tokenized["validation"], batch_size=64, shuffle=False, collate_fn=collator)

    # Load model and freeze base weights (keep classifier head trainable)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=cfg["num_labels"])
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id

    for name, param in model.named_parameters():
        if not any(head_name in name for head_name in ("classifier", "pooler", "score")):
            param.requires_grad_(False)

    # Covers DeBERTa, RoBERTa/BERT, and Qwen/Llama linear layer names
    target_modules = ("query_proj", "key_proj", "value_proj", "query", "key", "value", "q_proj", "k_proj", "v_proj", "o_proj", "dense")
    model = replace_linear_with_lplora(model, target_modules=target_modules, r=init_r, lora_alpha=init_r * 2)
    model.to(device)

    total_steps = len(train_loader) * epochs
    allocator = LpLoRAAllocator(
        model,
        total_steps=total_steps,
        M=M,
        p=p,
        prune_every=max(10, total_steps // 40),
        init_warmup_ratio=0.1,
        final_warmup_ratio=0.7,
        target_rank_ratio=target_ratio,
    )

    trainable_params = [param for param in model.parameters() if param.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps
    )

    epoch_history = []
    best_score = -1e9
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            if cfg["is_reg"]:
                batch["labels"] = batch["labels"].float()

            optimizer.zero_grad()
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                outputs = model(**batch)
                loss = outputs.loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            allocator.step()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        # Validation pass
        model.eval()
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
                logits = model(**batch).logits
            preds = logits.squeeze(-1) if cfg["is_reg"] else torch.argmax(logits, dim=-1)
            metric.add_batch(predictions=preds.cpu().numpy(), references=batch["labels"].cpu().numpy())

        eval_res = metric.compute()
        score = eval_res[cfg["metric"]] * 100.0
        best_score = max(best_score, score)

        ranks = allocator.get_rank_summary()
        alive_ranks = sum(ranks.values())
        avg_loss = total_loss / len(train_loader)

        epoch_history.append({
            "epoch": epoch,
            "train_loss": avg_loss,
            "val_score": score,
            "best_score": best_score,
            "alive_ranks": alive_ranks,
            "total_ranks": allocator.initial_total_rank,
        })

        print(
            f"Epoch {epoch:2d}/{epochs} | Loss: {avg_loss:.4f} | "
            f"Val {cfg['metric']}: {score:.2f}% (Best: {best_score:.2f}%) | "
            f"Ranks: {alive_ranks}/{allocator.initial_total_rank}"
        )

    final_layer_ranks = allocator.get_rank_summary()
    elapsed = time.time() - start_time

    # Store everything in global notebook log
    ALL_EXPERIMENT_LOGS[run_id] = {
        "model_name": model_name,
        "task": task,
        "metric_name": cfg["metric"],
        "best_score": best_score,
        "final_alive_ranks": sum(final_layer_ranks.values()),
        "initial_total_rank": allocator.initial_total_rank,
        "elapsed_sec": elapsed,
        "history": pd.DataFrame(epoch_history),
        "layer_ranks": final_layer_ranks,
    }

    # Clean up VRAM on the A4000 before next run
    del model, optimizer, scheduler, allocator
    gc.collect()
    torch.cuda.empty_cache()

    return ALL_EXPERIMENT_LOGS[run_id]

In [ ]:
# 1. Run DeBERTaV3-base on MRPC and RTE (from Table 1 of the paper)
run_lplora_experiment(model_name="microsoft/deberta-v3-base", task="mrpc", init_r=16, target_ratio=0.5, epochs=10, lr=8e-4)
run_lplora_experiment(model_name="microsoft/deberta-v3-base", task="rte",  init_r=16, target_ratio=0.5, epochs=10, lr=8e-4)

# 2. Compare against a second model architecture (e.g., RoBERTa-base) on MRPC
run_lplora_experiment(model_name="roberta-base", task="mrpc", init_r=16, target_ratio=0.5, epochs=10, lr=5e-4)

# Display Master Comparison Table
summary_rows = []
for run_id, data in ALL_EXPERIMENT_LOGS.items():
    summary_rows.append({
        "Run ID": run_id,
        "Model": data["model_name"],
        "Task": data["task"].upper(),
        "Metric": data["metric_name"],
        "Best Val Score (%)": round(data["best_score"], 2),
        "Final Ranks": f"{data['final_alive_ranks']} / {data['initial_total_rank']}",
        "Time (s)": round(data["elapsed_sec"], 1),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)